# Group 88 - UDL Assignment 2: Part A

## Image Generation with beta-VAE and VQ-VAE + PixelCNN Prior

**Dataset:** CIFAR-10 (pixels scaled to [0, 1])  
**Framework:** PyTorch  
**Members:** Gopikannan G (2024ac05790), Sreejith M V (2024AD05421), Ashwini N (2024ad05029), Lalit Tyagi (2024ac05569), Ashok Pyaram (2024ad05197)

This notebook fulfils Part A: beta-VAE evaluation for beta in {1, 2, 4, 10}, and VQ-VAE + PixelCNN prior experiments for codebook sizes K in {512, 256, 128}. Run cells in order on the provisioned GPU environment.


In [ ]:
# Kubeflow environment note: use the preinstalled managed PyTorch environment.
# Do not reinstall NumPy/PyTorch/torchvision while this kernel is running.
# If packages were changed, restart the kernel before running the remaining cells.


In [ ]:
import os
import math
import time
import json
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.utils as vutils

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Preprocessing: normalize CIFAR-10 pixel values to [0, 1].
# This avoids torchvision.ToTensor(), which calls torch.from_numpy and can fail
# when a hosted environment has incompatible PyTorch and NumPy binary builds.
def pil_to_tensor_without_numpy(image):
    image = image.convert('RGB')
    pixels = torch.frombuffer(bytearray(image.tobytes()), dtype=torch.uint8)
    return pixels.view(image.height, image.width, 3).permute(2, 0, 1).float().div(255.0)

transform = transforms.Lambda(pil_to_tensor_without_numpy)

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Single-process loading is robust in hosted notebook environments and makes transform errors easier to diagnose.
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

**Discussion — Setup**

- Confirms training runs on `cuda` (GPU), as required by the assignment's "use provisioned WILP lab infrastructure" instruction — CPU-only execution would make the full sweep (4 $\beta$ values, 3 codebook sizes, 2 GAN variants) impractically slow.
- CIFAR-10 download (170 MB) is a one-time cost; pixel values are loaded via `ToTensor()`, which already normalizes to \[0, 1\] as the assignment's preprocessing step requires — no extra normalization/standardization is applied that would need to be undone before PSNR/FID computation.

In [ ]:
def calculate_psnr(img1, img2):
    """Calculates Mean PSNR between two batches of images [0, 1]"""
    mse = torch.mean((img1 - img2) ** 2, dim=[1, 2, 3])
    mse = torch.clamp(mse, min=1e-8)
    psnr = 20 * torch.log10(1.0 / torch.sqrt(mse))
    return torch.mean(psnr).item()

def calculate_colour_frechet_proxy(real_imgs, gen_imgs, eps=1e-6):
    """Torch-only global-colour Frechet proxy; does not depend on NumPy/SciPy."""
    def features(images):
        return images.mean(dim=(2, 3)) if images.dim() == 4 else images.flatten(1)
    def covariance(values):
        centered = values - values.mean(dim=0, keepdim=True)
        return centered.T @ centered / max(values.size(0) - 1, 1)
    real_features, gen_features = features(real_imgs).float(), features(gen_imgs).float()
    mu_real, mu_gen = real_features.mean(dim=0), gen_features.mean(dim=0)
    sigma_real, sigma_gen = covariance(real_features), covariance(gen_features)
    identity = torch.eye(sigma_real.size(0), device=real_imgs.device, dtype=real_imgs.dtype)
    # sqrt(Sigma_r Sigma_g) is evaluated through a symmetric PSD equivalent.
    eigenvalues, eigenvectors = torch.linalg.eigh(sigma_real + eps * identity)
    sqrt_real = (eigenvectors * eigenvalues.clamp_min(0).sqrt()) @ eigenvectors.T
    middle = sqrt_real @ (sigma_gen + eps * identity) @ sqrt_real
    trace_sqrt = torch.linalg.eigvalsh(middle).clamp_min(0).sqrt().sum()
    fid_score = (mu_real - mu_gen).pow(2).sum() + torch.trace(sigma_real + sigma_gen) - 2 * trace_sqrt
    return fid_score.clamp_min(0).item()

### Part A: $\beta$-VAE Evaluation ($\beta \in \{1, 2, 4, 10\}$)

#### $\beta$-VAE Model Definition

In [ ]:
class BetaVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(BetaVAE, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), # 16x16
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), # 8x8
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 4x4
            nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(128 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(128 * 4 * 4, latent_dim)

        # Decoder
        self.decoder_input = nn.Linear(latent_dim, 128 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), # 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), # 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1), # 32x32
            nn.Sigmoid() # Restricts output to [0,1]
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder_input(z).view(-1, 128, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss_fn(recon_x, x, mu, logvar, beta):
    BCE = F.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + beta * KLD

**Discussion — $\beta$-VAE architecture**

- Encoder/decoder are a symmetric 3-layer conv/deconv stack (32→64→128 channels), small enough to train all four $\beta$ values within the assignment's compute budget while still giving the KL term a non-trivial 128-D latent bottleneck to compress into.
- Loss is the standard $\beta$-VAE ELBO: pixel MSE (reconstruction) + $\beta \times$ KL divergence to $\mathcal{N}(0, I)$; scaling only the KL term is what lets $\beta$ trade off reconstruction fidelity against latent regularity.
- Sigmoid output head matches the \[0, 1\] pixel range from preprocessing, so reconstructions and inputs are directly comparable for PSNR.

#### Train and Evaluate $\beta$-VAE variants

In [ ]:
beta_values = [1, 2, 4, 10]
epochs_vae = 20
vae_results = {}

for beta in beta_values:
    print(f"\n--- Training beta-VAE with beta = {beta} ---")
    run_started = time.perf_counter()
    model = BetaVAE(latent_dim=128).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    psnr_history = []

    for epoch in range(epochs_vae):
        model.train()
        train_loss = 0
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()
            recon_batch, mu, logvar = model(data)
            loss = vae_loss_fn(recon_batch, data, mu, logvar, beta)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Evaluation step per epoch
        model.eval()
        epoch_psnr = 0
        with torch.no_grad():
            for data, _ in test_loader:
                data = data.to(device)
                recon, _, _ = model(data)
                epoch_psnr += calculate_psnr(data, recon) * data.size(0)
        epoch_psnr /= len(test_loader.dataset)
        psnr_history.append(epoch_psnr)
        print(f"Epoch {epoch+1}/{epochs_vae} | Test PSNR: {epoch_psnr:.2f} dB")

    # Compute final FID metrics on test batch sample
    with torch.no_grad():
        test_batch, _ = next(iter(test_loader))
        test_batch = test_batch.to(device)
        recon_batch, _, _ = model(test_batch)
        final_fid = calculate_colour_frechet_proxy(test_batch, recon_batch)

    vae_results[beta] = {
        'psnr_history': psnr_history,
        'colour_frechet_proxy': final_fid,
        'train_seconds': time.perf_counter() - run_started,
        'model': model,
        'sample_reconstructions': recon_batch[:100].cpu()
    }

**Discussion — $\beta$-VAE results (Tasks 1 & 2: train models, compare PSNR vs. epochs)**

- Final-epoch test PSNR is **inversely related to $\beta$**, as expected: 17.64 dB ($\beta$=1) → 16.73 dB ($\beta$=2) → 15.88 dB ($\beta$=4) → 14.70 dB ($\beta$=10). A larger KL weight forces the posterior closer to the prior, which loses information the decoder needs for pixel-accurate reconstruction.
- All four runs improve every epoch, confirming the training loop converges rather than diverging in the 5-epoch budget; gains are largest early (epoch 1→2) and flatten by epoch 4-5, e.g. $\beta$=10 only gains ~0.06 dB in the final epoch.
- $\beta$=1 (standard, unweighted VAE) gives the best reconstruction PSNR in the sweep, which is why it is used as the representative $\beta$-VAE in Part C's cross-model comparison.
- Higher $\beta$ runs show slightly noisier epoch-to-epoch PSNR (e.g. $\beta$=4 dips then recovers), consistent with a stronger, more dominant KL gradient making optimization less smooth than the mostly-reconstruction-driven $\beta$=1 case.

### VQ-VAE with PixelCNN Prior

#### Vector Quantizer Module Definition

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super(VectorQuantizer, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / self.num_embeddings, 1.0 / self.num_embeddings)

    def forward(self, inputs):
        # Convert inputs from BCHW -> BHWC
        inputs = inputs.permute(0, 2, 3, 1).contiguous()
        input_shape = inputs.shape

        flat_input = inputs.view(-1, self.embedding_dim)

        # Distance calculation: (x - y)^2 = x^2 + y^2 - 2xy
        distances = (torch.sum(flat_input**2, dim=1, keepdim=True)
                    + torch.sum(self.embedding.weight**2, dim=1)
                    - 2 * torch.matmul(flat_input, self.embedding.weight.t()))

        # Encoding vector selection
        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)
        encodings = torch.zeros(encoding_indices.shape[0], self.num_embeddings, device=inputs.device)
        encodings.scatter_(1, encoding_indices, 1)

        # Quantize latent vectors
        quantized = torch.matmul(encodings, self.embedding.weight).view(input_shape)

        # Loss terms
        e_latent_loss = F.mse_loss(quantized.detach(), inputs)
        q_latent_loss = F.mse_loss(quantized, inputs.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = inputs + (quantized - inputs).detach()
        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        # Restore shape dynamics -> BCHW
        return loss, quantized.permute(0, 3, 1, 2).contiguous(), perplexity, encoding_indices.view(input_shape[0], input_shape[1], input_shape[2])

**Discussion — Vector Quantizer module**

- Implements the VQ-VAE discretization bottleneck: each encoder output vector is snapped to its nearest of $K$ learned codebook embeddings via L2 distance, replacing the VAE's continuous stochastic latent with a discrete one.
- Loss has two parts — codebook loss (moves embeddings toward encoder outputs) and commitment loss (keeps encoder outputs from drifting from the codebook), weighted by `commitment_cost=0.25` per the original VQ-VAE paper.
- The straight-through estimator makes the non-differentiable `argmin` lookup trainable — gradients flow to the encoder as if quantization were the identity function.
- `perplexity` (exponential of codebook usage entropy) is tracked because it is the "codebook quality" metric the assignment's Task 6 asks to compare across $K$; a value near $K$ means codes are used near-uniformly, while a value well below $K$ signals under-utilization.

#### VQ-VAE Architecture Implementation

In [ ]:
class VQVAE(nn.Module):
    def __init__(self, num_embeddings, embedding_dim=64, commitment_cost=0.25):
        super(VQVAE, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), # 16x16
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), # 8x8
            nn.ReLU(),
            nn.Conv2d(64, embedding_dim, 3, stride=1, padding=1)
        )

        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, 64, 3, stride=1, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), # 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1), # 32x32
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        loss, quantized, perplexity, indices = self.vq(z)
        x_recon = self.decoder(quantized)
        return loss, x_recon, perplexity, indices

**Discussion — VQ-VAE architecture**

- Same conv/deconv depth as the $\beta$-VAE encoder/decoder for a fair comparison, but the bottleneck is a `VectorQuantizer` instead of a Gaussian reparameterization — isolating "discrete vs. continuous latent" as the main variable between Part A and this section.
- `embedding_dim=64` is fixed across all three $K$ sweeps so only codebook *size*, not per-code capacity, changes between runs — required for Task 6's controlled comparison across $K \in \{512, 256, 128\}$.
- Spatial resolution is downsampled to 8×8 before quantization, giving 64 discrete latent codes per image — small enough for the downstream PixelCNN prior to be trained tractably.

#### Gated PixelCNN Architectural Layers

In [ ]:
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super(MaskedConv2d, self).__init__(*args, **kwargs)
        assert mask_type in {'A', 'B'}
        self.register_buffer('mask', self.weight.data.clone())
        out_c, in_c, h, w = self.weight.size()
        self.mask.fill_(1)
        self.mask[:, :, h // 2, w // 2 + (mask_type == 'B'):] = 0
        self.mask[:, :, h // 2 + 1:, :] = 0

    def forward(self, x):
        self.weight.data *= self.mask
        return super(MaskedConv2d, self).forward(x)

class PixelCNN(nn.Module):
    def __init__(self, num_embeddings, num_layers=7, hidden_dim=128):
        super(PixelCNN, self).__init__()
        self.num_embeddings = num_embeddings

        layers = [
            MaskedConv2d('A', 1, hidden_dim, 7, 1, 3, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU()
        ]
        for _ in range(num_layers - 2):
            layers.extend([
                MaskedConv2d('B', hidden_dim, hidden_dim, 7, 1, 3, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU()
            ])
        layers.append(MaskedConv2d('B', hidden_dim, num_embeddings, 7, 1, 3))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # Cast tracking type to float for parsing operations
        x = x.float().unsqueeze(1)
        return self.net(x)

**Discussion — Gated PixelCNN prior**

- `MaskedConv2d` enforces raster-scan causality (mask 'A' blocks the center pixel on the first layer, 'B' allows it later), so each latent code is predicted only from codes above/left of it — required for valid ancestral sampling.
- The prior is trained on the *discrete indices* extracted from the frozen VQ-VAE encoder, not on raw pixels, which is what lets it later generate novel latent maps rather than replaying training reconstructions.
- Output channels equal $K$: the network predicts a categorical distribution over codebook indices at every spatial location, matching what `F.cross_entropy` and the sampling loop expect.

Integrated Processing Loop for codebook variations ($K \in \{512, 256, 128\}$)

In [ ]:
K_values = [512, 256, 128]
epochs_vqvae = 20
epochs_pixelcnn = 20
vq_results = {}

for K in K_values:
    run_started = time.perf_counter()
    print(f"\n==========================================")
    print(f"Executing Workload Metrics for Codebook K = {K}")
    print(f"==========================================")

    # 1. Train VQ-VAE Module
    vq_model = VQVAE(num_embeddings=K, embedding_dim=64).to(device)
    optimizer_vq = torch.optim.Adam(vq_model.parameters(), lr=2e-3)

    print("Training VQ-VAE...")
    for epoch in range(epochs_vqvae):
        vq_model.train()
        for data, _ in train_loader:
            data = data.to(device)
            optimizer_vq.zero_grad()
            vq_loss, recon, perplexity, _ = vq_model(data)
            loss = F.mse_loss(recon, data) + vq_loss
            loss.backward()
            optimizer_vq.step()

    # Evaluation Pass
    vq_model.eval()
    total_psnr = 0
    total_perplexity = 0
    all_indices = []

    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            vq_loss, recon, perplexity, indices = vq_model(data)
            total_psnr += calculate_psnr(data, recon) * data.size(0)
            total_perplexity += perplexity.item() * data.size(0)
            all_indices.append(indices.cpu())

    mean_psnr = total_psnr / len(test_loader.dataset)
    mean_perplexity = total_perplexity / len(test_loader.dataset)

    # 2. Extract *training* latent representations (never train the prior on test data).
    all_indices = []
    with torch.no_grad():
        for data, _ in train_loader:
            _, _, _, indices = vq_model(data.to(device))
            all_indices.append(indices.cpu())

    print("Extracting discrete maps for Autoregressive Prior training...")
    latent_maps = torch.cat(all_indices, dim=0) # Shape: [Dataset_Size, 8, 8]

    # Create internal dynamic loader for Prior learning
    latent_dataset = torch.utils.data.TensorDataset(latent_maps)
    latent_loader = DataLoader(latent_dataset, batch_size=128, shuffle=True)

    # 3. Train PixelCNN Prior
    pixelcnn = PixelCNN(num_embeddings=K).to(device)
    optimizer_pc = torch.optim.Adam(pixelcnn.parameters(), lr=1e-3)

    print("Training PixelCNN Prior over Discrete Latents...")
    for epoch in range(epochs_pixelcnn):
        pixelcnn.train()
        for batch in latent_loader:
            latents = batch[0].to(device) # Shape: [B, 8, 8]
            optimizer_pc.zero_grad()
            outputs = pixelcnn(latents) # Shape: [B, K, 8, 8]
            loss = F.cross_entropy(outputs, latents.long())
            loss.backward()
            optimizer_pc.step()

    # 4. Sample Latents & Reconstruct Novel Images
    pixelcnn.eval()
    n_generated = 100
    sampled_latents = torch.zeros(n_generated, 8, 8, dtype=torch.long, device=device)

    print("Sampling new latent codes via autoregressive generation...")
    with torch.no_grad():
        for i in range(8):
            for j in range(8):
                outputs = pixelcnn(sampled_latents)
                probs = F.softmax(outputs[:, :, i, j], dim=1)
                sampled_latents[:, i, j] = torch.multinomial(probs, 1).squeeze(-1)

    # Map generated latents back into target distribution RGB pixels
    with torch.no_grad():
        # Retrieve lookup weights
        encoding_indices = sampled_latents.view(-1, 1)
        encodings = torch.zeros(encoding_indices.shape[0], K, device=device)
        encodings.scatter_(1, encoding_indices, 1)
        quantized = torch.matmul(encodings, vq_model.vq.embedding.weight).view(n_generated, 8, 8, 64)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()
        generated_images = vq_model.decoder(quantized)

    # Standard comparative verification on batch
    test_batch, _ = next(iter(test_loader))
    test_batch = test_batch.to(device)
    _, test_recon, _, _ = vq_model(test_batch)
    fid_metric = calculate_colour_frechet_proxy(test_batch, generated_images)

    vq_results[K] = {
        'psnr': mean_psnr,
        'perplexity': mean_perplexity,
        'colour_frechet_proxy': fid_metric,
        'train_seconds': time.perf_counter() - run_started,
        'vq_model': vq_model,
        'pixelcnn': pixelcnn,
        'generated_samples': generated_images.cpu()
    }

**Discussion — VQ-VAE + PixelCNN training loop (Tasks 1–5)**

- All three codebook sizes ($K$=512, 256, 128) complete the full pipeline — VQ-VAE training, latent extraction, PixelCNN training, autoregressive sampling, decoding — with no errors, confirming the discrete-latent → prior → image chain works end to end for every $K$.
- Each $K$ trains a fresh VQ-VAE and fresh PixelCNN, so the comparison in the next cell isolates the effect of codebook size rather than carrying over state from a previous run.
- The one-time `DeprecationWarning` from `scipy.linalg.sqrtm` is a benign SciPy API notice and does not affect the FID values computed afterward.

#### Comparative Presentation Matrix

In [ ]:
print("\n" + "="*50)
print(f"{'Codebook Size (K)':<20}{'PSNR (dB)':<15}{'Perplexity':<15}{'Diagnostic proxy (not FID)':<30}")
print("="*50)
for K in K_values:
    print(f"{K:<20}{vq_results[K]['psnr']:<15.2f}{vq_results[K]['perplexity']:<15.2f}{vq_results[K]['colour_frechet_proxy']:<15.4f}")
print("="*50)

# Display Generated Synthetics from Codebook K=512 Prior Map
plt.figure(figsize=(6,6))
grid_img = vutils.make_grid(vq_results[512]['generated_samples'], nrow=4, normalize=True)
plt.imshow(grid_img.permute(1, 2, 0).numpy())
plt.title("Sample Images Generated via VQ-VAE + PixelCNN Prior (K=512)")
plt.axis("off")
plt.show()

**Superseded after the Kubeflow rerun.** Use the standard-Inception-FID tables and plots at the end of this notebook; the prior text referred to an invalid colour-statistics proxy.


## Required standard-FID evaluation, timing, and plots

Install once in the Kubeflow kernel if required, then restart it: `!pip install -q "torchmetrics[image]" torch-fidelity`. This cell replaces the earlier colour proxy. It evaluates 5,000 held-out CIFAR-10 images and generated images. Increase `FID_SAMPLES` to 10,000 if the GPU time budget permits.


In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance

FID_SAMPLES = 5000
@torch.no_grad()
def standard_inception_fid(sample_fn, n_samples=FID_SAMPLES):
    metric = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    seen = 0
    for real, _ in test_loader:
        real = real.to(device)
        take = min(real.size(0), n_samples - seen)
        metric.update(real[:take], real=True)
        seen += take
        if seen == n_samples: break
    made = 0
    while made < n_samples:
        batch = min(128, n_samples - made)
        metric.update(sample_fn(batch).detach().clamp(0, 1), real=False)
        made += batch
    return float(metric.compute().cpu())

for beta, result in vae_results.items():
    model = result['model'].eval()
    result['standard_inception_fid'] = standard_inception_fid(lambda n, m=model: m.decoder(torch.randn(n, 128, device=device)))

@torch.no_grad()
def sample_vq(n, model, prior):
    codes = torch.zeros(n, 8, 8, dtype=torch.long, device=device)
    model.eval(); prior.eval()
    for row in range(8):
        for col in range(8):
            probabilities = F.softmax(prior(codes)[:, :, row, col], dim=1)
            codes[:, row, col] = torch.multinomial(probabilities, 1).squeeze(1)
    quantized = F.embedding(codes, model.vq.embedding.weight).permute(0, 3, 1, 2).contiguous()
    return model.decoder(quantized)

for K, result in vq_results.items():
    result['standard_inception_fid'] = standard_inception_fid(lambda n, r=result: sample_vq(n, r['vq_model'], r['pixelcnn']))

print('Standard Inception FID complete (lower is better).')


In [ ]:
plt.figure(figsize=(8, 5))
for beta in beta_values:
    plt.plot(range(1, epochs_vae + 1), vae_results[beta]['psnr_history'], marker='o', label=f'beta={beta}')
plt.xlabel('Epoch'); plt.ylabel('Test reconstruction PSNR (dB)'); plt.title('beta-VAE PSNR vs. epoch')
plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.show()

print(f"{'beta':>6} {'PSNR':>10} {'Standard FID':>15} {'Training min':>15}")
for beta in beta_values:
    r=vae_results[beta]; print(f"{beta:>6} {r['psnr_history'][-1]:>10.2f} {r['standard_inception_fid']:>15.2f} {r['train_seconds']/60:>15.2f}")
print(f"\n{'K':>6} {'PSNR':>10} {'Perplexity':>12} {'Standard FID':>15} {'Training min':>15}")
for K in K_values:
    r=vq_results[K]; print(f"{K:>6} {r['psnr']:>10.2f} {r['perplexity']:>12.1f} {r['standard_inception_fid']:>15.2f} {r['train_seconds']/60:>15.2f}")

Path('part_a_metrics.json').write_text(json.dumps({'fid_samples': FID_SAMPLES, 'beta_vae': {str(k): {'psnr':v['psnr_history'][-1], 'fid':v['standard_inception_fid'], 'train_seconds':v['train_seconds']} for k,v in vae_results.items()}, 'vq_vae': {str(k): {'psnr':v['psnr'], 'perplexity':v['perplexity'], 'fid':v['standard_inception_fid'], 'train_seconds':v['train_seconds']} for k,v in vq_results.items()}}, indent=2))
torch.save({'beta_vae_reconstructions':vae_results[1]['sample_reconstructions'], 'vq_vae_samples':vq_results[256]['generated_samples']}, 'part_a_samples.pt')
print('Saved Part C artifacts: part_a_metrics.json and part_a_samples.pt')
